# Solar filament segmentation: Mask R-CNN pipeline (final submission)

This notebook runs the full pipeline used for our final Kaggle submission (public LB 0.35):

1. environment and data paths
2. pre-processing (grayscale -> 3 channels: raw / CLAHE / high-pass)
3. loading the released weights from `weights/mask_rcnn_v5_best.pt` (no re-training needed)
4. validation on the held-out 20% split (home PQ)
5. inference on `test_images` with 8-view TTA and the instance selector -> `submission_mask_rcnn_v5.csv`
6. visual check of a few predictions
7. (optional) re-training from scratch

Runs on Kaggle (GPU T4, Add Data = the competition) or locally with `MAGFILO_ROOT` pointing to `MAGFiLO_1.0_Kaggle_2026/`.

## 1. Environment

In [ ]:
import os, sys, subprocess, hashlib
from pathlib import Path

# repo root: works when the notebook is opened from notebooks/ or from the repo root
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir() and Path("/kaggle/working/filament-segmentation-2026").is_dir():
    ROOT = Path("/kaggle/working/filament-segmentation-2026")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("repo:", ROOT)

# On Kaggle torch/torchvision are preinstalled; this installs the rest.
INSTALL = False
if INSTALL:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

In [ ]:
import numpy as np
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| cuda", torch.cuda.is_available())

from src.config import config as project_config
print("dataset root:", project_config.dataset_path)
print("train images:", project_config.train_images.is_dir(), "| test images:", project_config.test_images.is_dir())

## 2. Pre-processing

Each 2048x2048 grayscale H-alpha image is turned into a 3-channel input: raw intensity, CLAHE-equalised intensity and a high-pass layer (image minus its Gaussian blur). The network rescales the input to 1344x1344 internally (`min_size = max_size = 1344`); predicted masks are returned at the original 2048x2048 resolution.

In [ ]:
import cv2
import matplotlib.pyplot as plt
from src.data import three_channel

sample = sorted(project_config.test_images.iterdir())[0]
gray = cv2.imread(str(sample), cv2.IMREAD_GRAYSCALE)
x3 = three_channel(gray)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, ch, name in zip(axes, range(3), ["raw", "CLAHE", "high-pass"]):
    ax.imshow(x3[..., ch], cmap="gray"); ax.set_title(name); ax.axis("off")
plt.suptitle(sample.name); plt.tight_layout(); plt.show()

## 3. Released weights

GitHub limits files to 100 MB, so the 188 MB checkpoint is stored as four chunks `weights/mask_rcnn_v5_best.pt.part-00..03`. `weights/assemble.py` joins them into `weights/mask_rcnn_v5_best.pt` and checks the SHA-256.

In [ ]:
sys.path.insert(0, str(ROOT / "weights"))
from assemble import assemble, SHA256

WEIGHTS = assemble()
print("weights OK:", WEIGHTS, f"{WEIGHTS.stat().st_size / 1e6:.1f} MB", "sha256", SHA256[:12])

In [ ]:
from src.models.mask_rcnn.config import config as mcfg
from src.models.mask_rcnn.infer import load_checkpoint, predict_masks

ckpt = torch.load(WEIGHTS, map_location="cpu", weights_only=False)
print({k: ckpt[k] for k in ("epoch", "image_size", "architecture")})
print("stored metrics:", {k: ckpt["metrics"].get(k) for k in ("pq", "tp", "fp", "fn")})
del ckpt

model, device = load_checkpoint(WEIGHTS)
print("model on", device, "| params:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

## 4. Validation (home PQ)

The validation split is 20% of image file names (seed 42), so all annotator copies of one observation stay on the same side. Matching at IoU >= 0.5, PQ = sum(IoU) / (TP + 0.5 FP + 0.5 FN), summed over whole images.
Full validation (387 images) with full TTA takes about 1 h on a T4; set `VAL_LIMIT` to a small number for a quick check.

In [ ]:
from tqdm.auto import tqdm
from src.metrics import aggregate, panoptic_quality
from src.models.mask_rcnn.dataset import FilamentMaskRCNNDataset

RUN_VAL = True
VAL_LIMIT = 30          # 0 = all 387 validation images
VAL_TTA = "full"        # False | "light" | "full"

if RUN_VAL:
    val_ds = FilamentMaskRCNNDataset(train=False, augment=False)
    rows = []
    n = len(val_ds) if not VAL_LIMIT else min(VAL_LIMIT, len(val_ds))
    for i in tqdm(range(n), desc="val"):
        image, target = val_ds[i]
        pred = predict_masks(model, image, device, tta=VAL_TTA)
        gt = [m.numpy().astype("uint8") for m in target["masks"]]
        rows.append(panoptic_quality(gt, pred))
    print(aggregate(rows))

## 5. Test inference -> submission

Post-processing (defaults in `src/models/mask_rcnn/config.py`): 8-view TTA (rotations, flips, transposes), score >= 0.55, mask threshold 0.5, area >= 150 px, weighted merge of duplicate candidates (mask IoU 0.6), then pixel-level arbitration so that masks never overlap. This reproduces `submission_mask_rcnn_v5.csv` (1760 masks for 180 test images); about 35 min on a T4.

In [ ]:
from src.models.mask_rcnn.submission import write_submission

OUT = Path(project_config.outputs_path) / "submission_mask_rcnn_v5.csv"
write_submission(WEIGHTS, output=OUT, tta="full")

import pandas as pd
sub = pd.read_csv(OUT)
print(sub.shape, sub["filament_id"].str.rsplit("_", n=1).str[0].nunique(), "images")
sub.head()

## 6. Visual check

In [ ]:
from pycocotools import mask as mask_utils

def decode(counts):
    return mask_utils.decode({"size": [2048, 2048], "counts": counts.encode()})

stems = sub["filament_id"].str.rsplit("_", n=1).str[0]
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, stem in zip(axes, stems.drop_duplicates().iloc[:3]):
    path = next(project_config.test_images.glob(stem + ".*"))
    img = cv2.cvtColor(cv2.imread(str(path), cv2.IMREAD_GRAYSCALE), cv2.COLOR_GRAY2RGB)
    for counts in sub.loc[stems == stem, "segmentation_rle"]:
        m = decode(counts)
        contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(img, contours, -1, (255, 60, 60), 4)
    ax.imshow(img); ax.set_title(f"{stem}: {(stems == stem).sum()} filaments"); ax.axis("off")
plt.tight_layout(); plt.show()

## 7. (Optional) training from scratch

The released checkpoint was trained on Kaggle (1x T4, ~8 h): 42 epochs, AdamW lr 1.5e-4 with 2-epoch warm-up and cosine decay to 5e-6, weight decay 1e-4, grad clip 5, AMP, batch size 1, random 90-degree rotations / flips and brightness / contrast / gamma / noise jitter. Initialisation: weights of our previous Mask R-CNN v2 run (`init_from`), which we do not release; training with `init_from=None` starts from random weights. The best epoch is selected by full-validation PQ (epoch 36 for the released file).

In [ ]:
TRAIN = False
if TRAIN:
    from src.models.mask_rcnn.train import train
    mcfg.output_dir = Path(project_config.outputs_path) / "mask_rcnn"
    best = train(epochs=mcfg.epochs, image_size=mcfg.image_size, eval_limit=50, init_from=None)
    print(best)